# Exploratory Data Analysis

Análisis exploratorio de los datos utilizados en el proceso de limpieza.

El objetivo es revisar la estructura de los datasets, identificar valores faltantes,
analizar variables categóricas y numéricas y observar posibles relaciones entre
las variables antes de definir las transformaciones de limpieza.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, desc, regexp_replace

In [ ]:
spark = SparkSession.builder.appName("FinancialTransactions_EDA").getOrCreate()

## Carga de los datos

Se cargan los datasets utilizando PySpark para mantener el procesamiento
dentro del mismo entorno utilizado para la limpieza de datos.

In [ ]:
cards_spark = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/cards_data.csv")
)

transactions_spark = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/transactions_data.csv")
)

users_spark = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv("../data/modified/users_data.csv")
)

## Estructura de los datasets

Primero se revisa el tamaño y los tipos de datos de cada conjunto.

In [ ]:
print("Users:")
print(f"Filas: {users_spark.count()}")
print(f"Columnas: {len(users_spark.columns)}")

users_spark.printSchema()

In [ ]:
print("Cards:")
print(f"Filas: {cards_spark.count()}")
print(f"Columnas: {len(cards_spark.columns)}")

cards_spark.printSchema()

In [ ]:
print("Transactions:")
print(f"Filas: {transactions_spark.count()}")
print(f"Columnas: {len(transactions_spark.columns)}")

transactions_spark.printSchema()

## Valores faltantes

Se revisan los valores nulos de cada dataset para identificar columnas que puedan requerir algún tratamiento durante la limpieza.

In [ ]:
def null_summary(df):
    return df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])

In [ ]:
print("Users")
null_summary(users_spark).show()

In [ ]:
print("Cards")
null_summary(cards_spark).show()

In [ ]:
print("Transactions")
null_summary(transactions_spark).show()

## Variables categóricas

Se revisan las categorías presentes en algunas variables para conocer su
distribución y detectar valores poco frecuentes o inconsistentes.

In [ ]:
cards_spark.groupBy("card_type").count().orderBy(desc("count")).show()

In [ ]:
cards_spark.groupBy("card_brand").count().orderBy(desc("count")).show()

In [ ]:
print("Tipos de tarjeta:", cards_spark.select("card_type").distinct().count())

print("Marcas de tarjeta:", cards_spark.select("card_brand").distinct().count())

## Credit Score

Se revisa la distribución de los valores de credit score mediante estadística descriptiva. Esto permite observar el rango de valores y su comportamiento general antes de realizar transformaciones sobre los datos.

In [ ]:
users_spark.select("credit_score").summary().show()

In [ ]:
credit_score_distribution = (
    users_spark.select("credit_score")
    .dropna()
    .groupBy("credit_score")
    .count()
    .orderBy("credit_score")
)

credit_score_distribution.show()

In [ ]:
import matplotlib.pyplot as plt

credit_score_pd = credit_score_distribution.toPandas()

plt.figure(figsize=(10, 5))
plt.hist(credit_score_pd["credit_score"], weights=credit_score_pd["count"], bins=20)

plt.title("Distribución de Credit Score")
plt.xlabel("Credit Score")
plt.ylabel("Frecuencia")
plt.show()

## Distribución de tipos de tarjeta

Se representa la frecuencia de cada categoría de card_type para observar
cómo se distribuyen los registros entre los diferentes tipos.

In [ ]:
card_type_distribution = cards_spark.groupBy("card_type").count().orderBy(desc("count"))

card_type_distribution.show()

In [ ]:
card_type_pd = card_type_distribution.toPandas()

card_type_pd["card_type"] = card_type_pd["card_type"].fillna("Missing")

plt.figure(figsize=(8, 5))

plt.bar(card_type_pd["card_type"], card_type_pd["count"])

plt.title("Distribución por tipo de tarjeta")
plt.xlabel("Tipo de tarjeta")
plt.ylabel("Frecuencia")

plt.show()

## Correlaciones

Se analizan relaciones entre variables numéricas de los usuarios y sus tarjetas. Para relacionar el credit score con el límite de crédito se combinan ambos datasets mediante el identificador del cliente.

In [ ]:
users_cards_spark = users_spark.join(
    cards_spark, users_spark["id"] == cards_spark["client_id"], "inner"
)

In [ ]:
credit_score_limit_corr = users_cards_spark.select(
    col("credit_score").cast("double").alias("credit_score"),
    regexp_replace(col("credit_limit"), "[$,]", "")
    .cast("double")
    .alias("credit_limit"),
).stat.corr("credit_score", "credit_limit")

print(
    f"Correlación entre Credit Score y Credit Limit: " f"{credit_score_limit_corr:.4f}"
)

In [ ]:
credit_score_age_corr = users_spark.stat.corr("credit_score", "current_age")

print(f"Correlación entre Credit Score y Current Age: " f"{credit_score_age_corr:.4f}")

## Explorar cuentas, obligaciones y movimientos

Se revisa la información de cuentas y obligaciones de pago.

También se identifica la cobertura temporal de los movimientos y
su distribución por tipo.

Los campos de tarjeta y comercio pueden permanecer vacíos cuando
no corresponden al tipo de movimiento.

In [ ]:
# Importar librerías

from pyspark.sql import functions as F

# Cargar datasets adicionales

accounts_spark = spark.read.option("header", True).csv(
    "../data/modified/accounts_data.csv"
)

obligations_spark = spark.read.option("header", True).csv(
    "../data/modified/obligations_payments_data.csv"
)


# Revisar estructura y valores faltantes

for nombre, datos in [("Accounts", accounts_spark), ("Obligations", obligations_spark)]:
    print(nombre)
    print("Filas:", datos.count())

    datos.printSchema()
    datos.show(5, truncate=False)

    null_summary(datos).show(truncate=False)


# Revisar convenciones de importes

transactions_spark.groupBy("amount_convention").count().show()


# Revisar tipos de movimiento

transactions_spark.groupBy("movement_type", "direction", "currency").count().orderBy(
    F.desc("count")
).show(truncate=False)


# Revisar cobertura mensual

movements_eda = (
    transactions_spark.filter(F.col("amount_convention") == "signed_cashflow")
    .withColumn(
        "movement_date",
        F.try_to_timestamp(F.col("date").cast("string"), F.lit("yyyy-MM-dd HH:mm:ss")),
    )
    .withColumn("month", F.trunc(F.to_date("movement_date"), "month"))
)

movements_eda.groupBy("month").agg(
    F.count("*").alias("total_movements"),
    F.countDistinct("client_id").alias("total_clients"),
).orderBy("month").show(30, truncate=False)


# Revisar meses disponibles por cliente

movements_eda.groupBy("client_id").agg(
    F.countDistinct("month").alias("available_months")
).groupBy("available_months").count().orderBy("available_months").show()